# Yelp Fake-Detection Ablation — GPU Evaluation (Colab)

GPU version of `evaluate_yelp_ablation.py`. Evaluates the 6 models
(LSTM / DistilBERT × baseline / focal / contrastive) on the balanced test split with:
- decision threshold **re-tuned on the balanced val** (fair across models),
- threshold-free **PR-AUC / ROC-AUC**,
- confusion matrices, and the **~13% real-world deployment projection**.

**DistilBERT runs on the GPU** (the slow part on CPU); the LSTM stays on CPU (already fast).
Results match the local script (tiny GPU/CPU float differences are negligible).

### Setup
1. Runtime → Change runtime type → **GPU** (a free T4 is plenty — this is light inference).
2. Put the following in one Google Drive folder (default `MyDrive/yelp_eval/`):
   - `yelp_split.csv`, `yelp_split_imbalanced.csv` (the **deduped** versions)
   - the 6 folders: `yelp_fake_lstm_onnx/`, `yelp_fake_lstm_focal_onnx/`, `yelp_fake_lstm_contrastive_onnx/`,
     `yelp_fake_distilbert_onnx/`, `yelp_fake_distilbert_focal_onnx/`, `yelp_fake_distilbert_contrastive_onnx/`
3. Run all cells top to bottom. If the providers cell does not list `CUDAExecutionProvider`,
   restart the runtime (Runtime → Restart) and re-run from the install cell.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — set Runtime > Change runtime type > GPU.')

In [ ]:
# onnxruntime-gpu provides BOTH CPU and CUDA execution providers (import name: onnxruntime)
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q 'optimum[onnxruntime-gpu]' transformers
print('Installed. If the providers check below lacks CUDA, restart the runtime and re-run from here.')

In [ ]:
import onnxruntime as ort
print('ONNX Runtime providers:', ort.get_available_providers())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folder on Drive that holds the 6 _onnx folders + the 2 CSVs
DATA_DIR = "/content/drive/MyDrive/yelp_eval"

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    average_precision_score, roc_auc_score,
)
import onnxruntime as ort

REAL_WORLD_FAKE_RATE = 0.13
LSTM_MAX_LEN = 250   # must match LSTM training
BERT_MAX_LEN = 256   # must match DistilBERT training
BATCH_SIZE   = 64

LSTM_BASE_DIR        = os.path.join(DATA_DIR, "yelp_fake_lstm_onnx")
LSTM_FOCAL_DIR       = os.path.join(DATA_DIR, "yelp_fake_lstm_focal_onnx")
LSTM_CONTRASTIVE_DIR = os.path.join(DATA_DIR, "yelp_fake_lstm_contrastive_onnx")
BERT_BASE_DIR        = os.path.join(DATA_DIR, "yelp_fake_distilbert_onnx")
BERT_FOCAL_DIR       = os.path.join(DATA_DIR, "yelp_fake_distilbert_focal_onnx")
BERT_CONTRASTIVE_DIR = os.path.join(DATA_DIR, "yelp_fake_distilbert_contrastive_onnx")
SPLIT_CSV            = os.path.join(DATA_DIR, "yelp_split.csv")
IMBALANCED_CSV       = os.path.join(DATA_DIR, "yelp_split_imbalanced.csv")

_avail = ort.get_available_providers()
CUDA_OK = "CUDAExecutionProvider" in _avail
BERT_PROVIDER = "CUDAExecutionProvider" if CUDA_OK else "CPUExecutionProvider"
print("DistilBERT will run on:", "GPU (CUDA)" if CUDA_OK else "CPU (no CUDA EP!)")
if not CUDA_OK:
    print("WARNING: CUDA EP unavailable -> DistilBERT will be slow. Ensure GPU runtime + onnxruntime-gpu,")
    print("         then restart the runtime and re-run.")

# Preflight — fail fast if any required input is missing (report ALL)
_missing = []
for _f in (SPLIT_CSV, IMBALANCED_CSV):
    if not os.path.isfile(_f):
        _missing.append("  [missing file]    " + _f)
for _d in (LSTM_BASE_DIR, LSTM_FOCAL_DIR, LSTM_CONTRASTIVE_DIR,
           BERT_BASE_DIR, BERT_FOCAL_DIR, BERT_CONTRASTIVE_DIR):
    if not os.path.isdir(_d):
        _missing.append("  [missing folder]  " + _d)
    elif not any(fn.endswith(".onnx") for fn in os.listdir(_d)):
        _missing.append("  [no .onnx inside] " + _d)
if _missing:
    raise FileNotFoundError("Required inputs missing:\n" + "\n".join(_missing) +
                            "\n\nPut the 6 _onnx folders + 2 CSVs under " + DATA_DIR)
print("Preflight OK — all required inputs present.")

In [ ]:
# ── Load balanced val + test splits ───────────────────────────────
print("Loading balanced val + test splits...")
df      = pd.read_csv(SPLIT_CSV)
val_df  = df[df["split"] == "val" ].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
X_val,  y_val  = val_df["text"].tolist(),  val_df["label"].tolist()
X_test, y_test = test_df["text"].tolist(), test_df["label"].tolist()
print(f"  Val  rows : {len(X_val):,}  (genuine={sum(1 for y in y_val if y==0):,}  fake={sum(1 for y in y_val if y==1):,})")
print(f"  Test rows : {len(X_test):,}  (genuine={sum(1 for y in y_test if y==0):,}  fake={sum(1 for y in y_test if y==1):,})")

def _norm(t):
    return re.sub(r"\s+", " ", t.strip().lower())

# ── Leakage check — val + test must not appear in ablation training ──
print("\nLeakage check ...")
imb_df      = pd.read_csv(IMBALANCED_CSV)
train_norms = set(_norm(t) for t in imb_df[imb_df["split"] == "train"]["text"].tolist())
test_overlap = set(_norm(t) for t in X_test) & train_norms
val_overlap  = set(_norm(t) for t in X_val)  & train_norms
print(f"  Ablation training texts      : {len(train_norms):,}")
print(f"  Balanced TEST texts in train : {len(test_overlap):,}  (must be 0)")
print(f"  Balanced VAL  texts in train : {len(val_overlap):,}  (must be 0)")
if test_overlap or val_overlap:
    raise RuntimeError("LEAKAGE — regenerate yelp_split_imbalanced.csv.")
print("  PASS — clean.")

In [ ]:
# ── LSTM tokenizer (pure-Python Keras tokenizer, matches training) ──
class _KerasTokenizer:
    def __init__(self, word_index, filters, lower, num_words):
        self.word_index = word_index
        self._filters   = str.maketrans("", "", filters)
        self.lower      = lower
        self.num_words  = num_words
    @classmethod
    def from_json(cls, path):
        with open(path, encoding="utf-8") as f:
            cfg = json.load(f)["config"]
        wi = cfg["word_index"]
        if isinstance(wi, str):
            wi = json.loads(wi)
        return cls(wi, cfg["filters"], cfg.get("lower", True), cfg.get("num_words"))
    def texts_to_sequences(self, texts):
        seqs = []
        for text in texts:
            if self.lower:
                text = text.lower()
            text = text.translate(self._filters)
            seq = []
            for w in text.split():
                idx = self.word_index.get(w)
                if idx is not None and (self.num_words is None or idx < self.num_words):
                    seq.append(idx)
            seqs.append(seq)
        return seqs


# ── Metric helpers (identical to evaluate_yelp_ablation.py) ─────────
def tune_threshold(probs, y_true):
    best_t, best_f = 0.5, -1.0
    for t in np.arange(0.05, 0.96, 0.01):
        f = f1_score(y_true, (probs >= t).astype(int), average="macro", zero_division=0)
        if f > best_f:
            best_f, best_t = f, float(t)
    return round(best_t, 2), best_f

def row(name, y_true, y_pred):
    mf1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    fp  = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    fr  = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    ff1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gp  = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
    gr  = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    gf1 = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    tn, fpc, fnc, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tpr = tp / (tp + fnc) if (tp + fnc) else 0.0
    tnr = tn / (tn + fpc) if (tn + fpc) else 0.0
    return {"Model": name, "Macro F1": f"{mf1:.1%}",
            "Fake P": f"{fp:.1%}", "Fake R": f"{fr:.1%}", "Fake F1": f"{ff1:.1%}",
            "Genuine P": f"{gp:.1%}", "Genuine R": f"{gr:.1%}", "Genuine F1": f"{gf1:.1%}",
            "_macro_f1": mf1, "_tn": int(tn), "_fp": int(fpc), "_fn": int(fnc), "_tp": int(tp),
            "_tpr": tpr, "_tnr": tnr, "_fpr": 1.0 - tnr}

def assemble(label, probs_val, probs_test, stored_thr):
    pv, pt = np.asarray(probs_val, float), np.asarray(probs_test, float)
    yv, yt = np.asarray(y_val), np.asarray(y_test)
    thr, _ = tune_threshold(pv, yv)
    r = row(label, yt, (pt >= thr).astype(int))
    r["Thr*"]    = f"{thr:.2f}"
    r["PR-AUC"]  = f"{average_precision_score(yt, pt):.1%}"
    r["ROC-AUC"] = f"{roc_auc_score(yt, pt):.1%}"
    r["_pr_auc"] = average_precision_score(yt, pt)
    r["_thr"]    = thr
    r["_stored_thr"] = stored_thr
    return r

def project_precision(tpr, fpr, prior):
    denom = prior * tpr + (1.0 - prior) * fpr
    return (prior * tpr / denom) if denom > 0 else 0.0

def encode_lstm(texts, tok):
    padded = []
    for s in tok.texts_to_sequences(texts):
        s = s[:LSTM_MAX_LEN]
        s = s + [0] * (LSTM_MAX_LEN - len(s))
        padded.append(s)
    return np.array(padded, dtype=np.float32)

In [ ]:
# ── Model runners ──────────────────────────────────────────────────
def run_lstm(model_dir, label):
    """LSTM on CPU (already fast). Returns (probs_val, probs_test, stored_thr)."""
    base = os.path.basename(model_dir).replace("_onnx", "")
    onnx_path = os.path.join(model_dir, base + ".onnx")
    if not os.path.exists(onnx_path):
        onnx_path = os.path.join(model_dir, "model.onnx")
    tok_path = os.path.join(model_dir, base + "_tokenizer.json")
    if not os.path.exists(tok_path):
        for fn in os.listdir(model_dir):
            if fn.endswith("_tokenizer.json"):
                tok_path = os.path.join(model_dir, fn); break
    thresh_path = os.path.join(model_dir, base + "_threshold.json")
    if not os.path.exists(thresh_path):
        thresh_path = os.path.join(model_dir, "threshold.json")
    if not os.path.exists(onnx_path):
        print(f"  [{label}] MISSING — skipping"); return None

    tok = _KerasTokenizer.from_json(tok_path)
    stored_thr = 0.5
    if os.path.exists(thresh_path):
        with open(thresh_path) as f:
            stored_thr = json.load(f).get("threshold", 0.5)
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    iname = sess.get_inputs()[0].name

    def predict(texts):
        out = []
        for i in range(0, len(texts), BATCH_SIZE):
            out.extend(sess.run(None, {iname: encode_lstm(texts[i:i+BATCH_SIZE], tok)})[0].flatten().tolist())
        return out
    return predict(X_val), predict(X_test), stored_thr


def run_distilbert(model_dir, label):
    """DistilBERT ONNX on GPU via raw onnxruntime (no optimum)."""
    if not os.path.isdir(model_dir):
        print(f"  [{label}] MISSING - skipping"); return None
    from transformers import AutoTokenizer

    thresh_path = os.path.join(model_dir, "threshold.json")
    stored_thr = 0.5
    if os.path.exists(thresh_path):
        with open(thresh_path) as f:
            stored_thr = json.load(f).get("threshold", 0.5)

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if CUDA_OK else ["CPUExecutionProvider"]
    sess = ort.InferenceSession(os.path.join(model_dir, "model.onnx"), providers=providers)
    in_names = [i.name for i in sess.get_inputs()]

    def predict(texts):
        out = []
        for i in range(0, len(texts), BATCH_SIZE):
            enc = tokenizer(texts[i:i+BATCH_SIZE], padding=True, truncation=True,
                            max_length=BERT_MAX_LEN, return_tensors="np")
            feed = {n: enc[n].astype(np.int64) for n in in_names if n in enc}
            logits = sess.run(None, feed)[0]
            ex = np.exp(logits - logits.max(axis=1, keepdims=True))
            probs = (ex / ex.sum(axis=1, keepdims=True))[:, 1]   # P(fake)
            out.extend(probs.tolist())
        return out
    return predict(X_val), predict(X_test), stored_thr


In [ ]:
# ── Run all six models ─────────────────────────────────────────────
import time
rows = []

def evaluate(runner, model_dir, label):
    t0 = time.time()
    res = runner(model_dir, label)
    if res is None:
        return
    r = assemble(label, *res)
    rows.append(r)
    print(f"  [{label}] re-tuned thr={r['Thr*']} (shipped {res[2]:.2f})  "
          f"Macro F1={r['Macro F1']}  PR-AUC={r['PR-AUC']}  ({time.time()-t0:.0f}s)")

print("=" * 60); print("LSTM models (CPU)..."); print("=" * 60)
for d, lab in [(LSTM_BASE_DIR, "LSTM (baseline)"),
               (LSTM_FOCAL_DIR, "LSTM (focal)"),
               (LSTM_CONTRASTIVE_DIR, "LSTM (contrastive)")]:
    evaluate(run_lstm, d, lab)

print("\n" + "=" * 60); print("DistilBERT models (GPU)..."); print("=" * 60)
for d, lab in [(BERT_BASE_DIR, "DistilBERT (baseline)"),
               (BERT_FOCAL_DIR, "DistilBERT (focal)"),
               (BERT_CONTRASTIVE_DIR, "DistilBERT (contrastive)")]:
    evaluate(run_distilbert, d, lab)

In [ ]:
# ── Results table + deltas + confusion matrices + deployment projection ──
DISPLAY = ["Thr*", "Macro F1", "Fake P", "Fake R", "Fake F1",
           "Genuine P", "Genuine R", "Genuine F1", "PR-AUC", "ROC-AUC"]
tw = 28 + 11 * len(DISPLAY)

print("\n" + "=" * tw)
print("YELP FAKE DETECTION ABLATION  (n={:,}  genuine={:,}  fake={:,})  | threshold re-tuned on balanced val".format(
    len(y_test), sum(1 for y in y_test if y == 0), sum(1 for y in y_test if y == 1)))
print("=" * tw)

if not rows:
    print("No models evaluated.")
else:
    hdr = ["Model"] + DISPLAY
    w   = [28] + [11] * len(DISPLAY)
    print("".join(h.rjust(x) for h, x in zip(hdr, w)))
    print("-" * sum(w))
    for _, r in pd.DataFrame(rows).sort_values("_macro_f1", ascending=False).iterrows():
        print("".join(str(r[c]).rjust(x) for c, x in zip(hdr, w)))
    print("=" * tw)
    print("Thr* = threshold re-tuned on balanced val.  PR-AUC/ROC-AUC = threshold-free ranking quality.")

    by = {r["Model"]: r for r in rows}
    print("\nDelta summary (variant - baseline):")
    print(f"  {'':<28}{'dMacroF1':>11}{'dPR-AUC':>11}")
    for b, v, disp in [("LSTM (baseline)", "LSTM (focal)", "LSTM focal"),
                       ("LSTM (baseline)", "LSTM (contrastive)", "LSTM contrastive"),
                       ("DistilBERT (baseline)", "DistilBERT (focal)", "DistilBERT focal"),
                       ("DistilBERT (baseline)", "DistilBERT (contrastive)", "DistilBERT contrastive")]:
        if b in by and v in by:
            d1 = by[v]["_macro_f1"] - by[b]["_macro_f1"]
            d2 = by[v]["_pr_auc"]   - by[b]["_pr_auc"]
            print(f"  {disp:<28}{('+' if d1>=0 else '')+format(d1,'.1%'):>11}{('+' if d2>=0 else '')+format(d2,'.1%'):>11}")

    print("\n" + "=" * tw)
    print("CONFUSION MATRICES (balanced test, at Thr* — rows=actual, cols=predicted)")
    print("=" * tw)
    for r in sorted(rows, key=lambda x: x["_macro_f1"], reverse=True):
        print(f"\n  {r['Model']}  (Thr* = {r['Thr*']})")
        print(f"                    pred GENUINE   pred FAKE")
        print(f"    actual GENUINE   {r['_tn']:>11,}   {r['_fp']:>9,}")
        print(f"    actual FAKE      {r['_fn']:>11,}   {r['_tp']:>9,}")
        print(f"    Fake recall: {r['_tpr']:.1%}  |  False-positive rate: {r['_fpr']:.1%}  |  Genuine recall: {r['_tnr']:.1%}")

    pi = REAL_WORLD_FAKE_RATE
    nf, ng = int(round(1000 * pi)), 1000 - int(round(1000 * pi))
    print("\n" + "=" * tw)
    print(f"PROJECTED DEPLOYMENT AT ~{pi:.0%} FAKE  (per 1,000 reviews: {nf} fake / {ng} genuine)")
    print("=" * tw)
    h2 = ["Model", "Fake P@13%", "Fake R", "Fake F1@13%", "Flagged/1k", "TrueFake/1k", "FalseFlag/1k"]
    w2 = [28, 12, 10, 13, 12, 13, 14]
    print("".join(h.rjust(x) for h, x in zip(h2, w2)))
    print("-" * sum(w2))
    proj = []
    for r in rows:
        p = project_precision(r["_tpr"], r["_fpr"], pi)
        rec = r["_tpr"]
        f1 = (2 * p * rec / (p + rec)) if (p + rec) else 0.0
        tp1k, fp1k = nf * r["_tpr"], ng * r["_fpr"]
        proj.append((r["Model"], p, rec, f1, tp1k + fp1k, tp1k, fp1k))
    for m, p, rec, f1, fl, tp1k, fp1k in sorted(proj, key=lambda t: t[3], reverse=True):
        vals = [m, f"{p:.1%}", f"{rec:.1%}", f"{f1:.1%}", f"{fl:.0f}", f"{tp1k:.0f}", f"{fp1k:.0f}"]
        print("".join(v.rjust(x) for v, x in zip(vals, w2)))
    print("=" * tw)
    print("Fake P@13% = precision among flagged when only 13% are truly fake (deployment reality).")